# Vocult — Evaluation Metrics Notebook
**Purpose:** Evaluate the transcription/translation/dubbing pipeline with reference data and measurable audio-quality checks.

This notebook is designed to run from the Vocult repository root. It does **not** invent a single “overall accuracy” for an entire dubbing system: transcription, translation, speaker diarization, emotion recognition, and generated-audio quality need different metrics.

## What this notebook evaluates
- **ASR / transcription:** Word Error Rate (WER), Character Error Rate (CER)
- **Translation:** corpus BLEU and chrF (requires reference translations)
- **Speaker diarization:** DER when reference speaker labels and timed turns are available; a simple speaker-label accuracy option is also provided for aligned segments
- **Emotion classification:** accuracy, precision, recall, F1, confusion matrix, and classification report when ground-truth emotion labels exist
- **Generated audio:** duration drift, clipping rate, silence ratio, and optional objective similarity checks; these do not measure emotional naturalness by themselves.

> **Important:** You need human-verified reference labels/transcripts/translations to calculate supervised metrics. Without ground truth, report the objective diagnostics only and do not label them “accuracy.”

## 1. Setup
Run this notebook from the repository root (`D:/vocult` in the current screenshot). Install any missing packages in the same Python environment used by Vocult.

In [14]:
pip install matplotlib

  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
   ---------------------------------------- 0.0/9.3 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.3 MB 5.0 MB/s eta 0:00:02
   -------- ------------------------------- 2.1/9.3 MB 5.6 MB/s eta 0:00:02
   ---------------- ----------------------- 3.9/9.3 MB 6.3 MB/s eta 0:00:01
   ----------------------------- ---------- 6.8/9.3 MB 8.4 MB/s eta 0:00:01
   -------------------------------------- - 8.9/9.3 MB 8.7 MB/s eta 0:00:01
   ---------------------------------------- 9.3/9.3 MB 8.5 MB/s  0:00:01
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ---------------------------------------- 0.0/2.5 MB ? eta -:--:--
   ---------------------------------------- 2.5/2.5 MB 13.0 MB/s  0:00:00
   ---------------------------------------- 0.0/7.3 MB ? eta -:--:--
   ----------------- ---------------------- 3.1/7.3 MB 15.3 MB/s eta 0:00:01
   ---

ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'd:\\vocult\\.venv\\Lib\\site-packages\\contourpy\\util\\bokeh_renderer.py'
Check the permissions.


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
print("Working directory:", PROJECT_ROOT)
print("Top-level files:", [p.name for p in PROJECT_ROOT.iterdir()][:40])

# If this notebook is stored outside the repository, set the path manually:
# PROJECT_ROOT = Path(r"D:\\vocult")



Working directory: d:\vocult
Top-level files: ['.git', '.gitignore', '.venv', '.venv312', 'audio_extractor_worker.py', 'emotion_xtts-2.ipynb', 'files', 'main.py', 'README.md', 'transcribe_audio_worker.py', 'translate_text_worker.py', 'Vocult_Complete.ipynb', 'Vocult_Evaluation_Metrics.ipynb', 'xtts_worker-2.ipynb']


In [16]:
# Install only if needed, using the active notebook kernel:
# %pip install jiwer sacrebleu scikit-learn librosa soundfile matplotlib seaborn

import importlib.util

for package in ["jiwer", "sacrebleu", "sklearn", "librosa", "soundfile", "matplotlib"]:
    print(f"{package}: {'available' if importlib.util.find_spec(package) else 'NOT INSTALLED'}")


jiwer: available
sacrebleu: available
sklearn: available
librosa: available
soundfile: available
matplotlib: available


  Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl.metadata (3.8 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached matplotlib-3.11.2-cp312-cp312-win_amd64.whl (9.3 MB)
Using cached contourpy-1.4.0-cp312-cp312-win_amd64.whl (234 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)

   ------------- -------------------------- 1/3 [contourpy]
   ------------- -------------------------- 1/3 [contourpy]
   -------------------------- ------------- 2/3 [matplotlib]
   -------------------------- ------------- 2/3 [matplotlib]
   -------------------------- ------------- 2/3 [matplotlib]
   -------------------------- ------------- 2/3 [matplotlib]
   -------------------------- ------------- 2/3 [matplotlib]
   -------------------------- ------------- 2/3 [matplotlib]
   -------------------------- ------------- 2/3 [matplotlib]
   -------------------------- ------------- 2/3 [matpl


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Prepare evaluation data

Create a CSV such as `evaluation/evaluation_manifest.csv`. Each row should represent one source dialogue segment. Use empty cells for metrics you cannot yet evaluate.

Recommended columns:
- `sample_id`
- `audio_path` — source audio path (relative to `PROJECT_ROOT` or absolute)
- `dub_audio_path` — generated segment or full dubbed audio path
- `reference_transcript` — verified source-language transcript
- `predicted_transcript` — Vocult/WhisperX transcript
- `reference_translation` — human-verified target-language translation
- `predicted_translation` — NLLB output
- `reference_speaker` — human speaker label, e.g. `SPEAKER_00`
- `predicted_speaker` — diarization output for the same aligned segment
- `reference_emotion` — human-annotated emotion
- `predicted_emotion` — model-predicted emotion

Do not compare translations with different segmentation blindly. Normalize the segments so each prediction corresponds to the correct reference unit.

In [17]:
EVAL_DIR = PROJECT_ROOT / "evaluation"
EVAL_DIR.mkdir(exist_ok=True)
MANIFEST_PATH = EVAL_DIR / "evaluation_manifest.csv"

REQUIRED_COLUMNS = [
    "sample_id", "audio_path", "dub_audio_path",
    "reference_transcript", "predicted_transcript",
    "reference_translation", "predicted_translation",
    "reference_speaker", "predicted_speaker",
    "reference_emotion", "predicted_emotion",
]

if MANIFEST_PATH.exists():
    df = pd.read_csv(MANIFEST_PATH)
    print(f"Loaded {len(df)} rows from {MANIFEST_PATH}")
else:
    df = pd.DataFrame(columns=REQUIRED_COLUMNS)
    df.to_csv(MANIFEST_PATH, index=False)
    print(f"Created template: {MANIFEST_PATH}")
    print("Fill it with manually verified references and system predictions, then rerun this cell.")

for col in REQUIRED_COLUMNS:
    if col not in df.columns:
        df[col] = pd.NA

df.head()


Created template: d:\vocult\evaluation\evaluation_manifest.csv
Fill it with manually verified references and system predictions, then rerun this cell.


,sample_id,audio_path,dub_audio_path,reference_transcript,predicted_transcript,reference_translation,predicted_translation,reference_speaker,predicted_speaker,reference_emotion,predicted_emotion


## 3. ASR / transcription metrics — WER and CER

WER is the proportion of word-level edits required to transform the predicted transcript into the reference. CER does the same at character level. Lower is better; WER can exceed 1.0 when there are many insertions.

In [18]:
def nonempty_pairs(frame, reference_col, prediction_col):
    pairs = frame[[reference_col, prediction_col]].dropna()
    pairs = pairs[
        pairs[reference_col].astype(str).str.strip().ne("") &
        pairs[prediction_col].astype(str).str.strip().ne("")
    ]
    return pairs

asr_pairs = nonempty_pairs(df, "reference_transcript", "predicted_transcript")

if len(asr_pairs):
    try:
        from jiwer import wer, cer
        references = asr_pairs["reference_transcript"].astype(str).tolist()
        predictions = asr_pairs["predicted_transcript"].astype(str).tolist()
        print(f"Samples evaluated: {len(asr_pairs)}")
        print(f"WER: {wer(references, predictions):.4f}")
        print(f"CER: {cer(references, predictions):.4f}")
    except ImportError:
        print("Install jiwer first: %pip install jiwer")
else:
    print("No transcript reference/prediction pairs yet.")


No transcript reference/prediction pairs yet.


## 4. Translation metrics — BLEU and chrF

BLEU and chrF compare generated translations against human reference translations. They measure text overlap, not whether the translation preserves every nuance or emotion. Evaluate multiple references if available; this basic version uses one reference per segment.

In [ ]:
translation_pairs = nonempty_pairs(df, "reference_translation", "predicted_translation")

if len(translation_pairs):
    try:
        import sacrebleu
        refs = translation_pairs["reference_translation"].astype(str).tolist()
        preds = translation_pairs["predicted_translation"].astype(str).tolist()
        bleu = sacrebleu.corpus_bleu(preds, [refs])
        chrf = sacrebleu.corpus_chrf(preds, [refs])
        print(f"Segments evaluated: {len(translation_pairs)}")
        print(f"Corpus BLEU: {bleu.score:.2f}")
        print(f"Corpus chrF: {chrf.score:.2f}")
    except ImportError:
        print("Install sacrebleu first: %pip install sacrebleu")
else:
    print("No translation reference/prediction pairs yet.")


## 5. Speaker assignment metrics

For aligned segments with human speaker labels, report label accuracy, macro-F1, and a confusion matrix. **Caveat:** diarization speaker IDs are arbitrary (e.g. a system's `SPEAKER_00` may correspond to the human's `speaker_B`). Map predicted IDs to reference IDs consistently before using this simple score. For full diarization evaluation, use a DER implementation with reference and hypothesis RTTM files; DER accounts for missed speech, false alarm speech, and speaker confusion.

In [ ]:
speaker_pairs = nonempty_pairs(df, "reference_speaker", "predicted_speaker")

if len(speaker_pairs):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
    y_true = speaker_pairs["reference_speaker"].astype(str)
    y_pred = speaker_pairs["predicted_speaker"].astype(str)
    print(f"Aligned segments evaluated: {len(speaker_pairs)}")
    print(f"Speaker-label accuracy (only valid after ID mapping): {accuracy_score(y_true, y_pred):.4f}")
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    print(f"Macro precision: {p:.4f} | macro recall: {r:.4f} | macro F1: {f1:.4f}")
    print("\nClassification report:\n", classification_report(y_true, y_pred, zero_division=0))
    labels = sorted(set(y_true) | set(y_pred))
    display(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))
else:
    print("No speaker label pairs yet.")


## 6. Emotion metrics

Use this only when reference emotions are annotated by a human or a validated dataset. Define a consistent label set, for example `neutral`, `happy`, `sad`, `angry`, `fearful`, `disgusted`, `surprised`. If the system does not yet produce emotion labels, leave predicted labels empty and evaluate this stage later.

In [ ]:
emotion_pairs = nonempty_pairs(df, "reference_emotion", "predicted_emotion")

if len(emotion_pairs):
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
    y_true = emotion_pairs["reference_emotion"].astype(str).str.lower().str.strip()
    y_pred = emotion_pairs["predicted_emotion"].astype(str).str.lower().str.strip()
    labels = sorted(set(y_true) | set(y_pred))
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average="weighted", zero_division=0)
    print(f"Segments evaluated: {len(emotion_pairs)}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Macro precision / recall / F1: {p_macro:.4f} / {r_macro:.4f} / {f1_macro:.4f}")
    print(f"Weighted precision / recall / F1: {p_weighted:.4f} / {r_weighted:.4f} / {f1_weighted:.4f}")
    print("\nClassification report:\n", classification_report(y_true, y_pred, zero_division=0))
    import matplotlib.pyplot as plt
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm)
    ax.set_xticks(range(len(labels)), labels=labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)), labels=labels)
    ax.set_xlabel("Predicted emotion")
    ax.set_ylabel("Reference emotion")
    ax.set_title("Emotion Confusion Matrix")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    fig.tight_layout()
    plt.show()
else:
    print("No emotion reference/prediction pairs yet.")


## 7. Audio-level diagnostics

These diagnostics are useful for the current issues (overlapping voices, timing problems, silence, clipping). They are **not** emotion-preservation accuracy metrics.

- **Duration drift:** dubbed duration minus source duration, in seconds and percentage.
- **Clipping ratio:** fraction of samples close to full-scale amplitude.
- **Silence ratio:** rough proportion of frames with very low RMS energy.
- **Peak amplitude / RMS:** basic signal-level checks.

For meaningful overlap detection, segment each speaker separately or use diarization timestamps and inspect whether generated segments collide on the timeline. A single mixed waveform cannot reliably tell whether two voices overlap.

In [ ]:
try:
    import soundfile as sf
    import librosa
except ImportError:
    sf = None
    librosa = None
    print("Install audio packages: %pip install soundfile librosa")

def resolve_path(value):
    if pd.isna(value) or not str(value).strip():
        return None
    p = Path(str(value))
    return p if p.is_absolute() else PROJECT_ROOT / p

def audio_diagnostics(path, silence_rms_threshold=0.01, clipping_threshold=0.999):
    path = Path(path)
    if not path.exists():
        return {"error": f"File not found: {path}"}
    if sf is None:
        return {"error": "soundfile is not installed"}
    audio, sr = sf.read(path, always_2d=True)
    mono = audio.mean(axis=1)
    if len(mono) == 0:
        return {"error": "Empty audio"}
    frame_len = max(1, int(sr * 0.025))
    n_frames = int(np.ceil(len(mono) / frame_len))
    padded = np.pad(mono, (0, n_frames * frame_len - len(mono)))
    frames = padded.reshape(n_frames, frame_len)
    frame_rms = np.sqrt(np.mean(frames ** 2, axis=1))
    duration = len(mono) / sr
    return {
        "path": str(path),
        "sample_rate_hz": sr,
        "duration_seconds": duration,
        "channels": audio.shape[1],
        "peak_amplitude": float(np.max(np.abs(audio))),
        "rms_amplitude": float(np.sqrt(np.mean(audio ** 2))),
        "clipping_ratio": float(np.mean(np.abs(audio) >= clipping_threshold)),
        "silence_ratio": float(np.mean(frame_rms < silence_rms_threshold)),
    }

audio_rows = []
for _, row in df.iterrows():
    source_path = resolve_path(row.get("audio_path", pd.NA))
    dub_path = resolve_path(row.get("dub_audio_path", pd.NA))
    source_diag = audio_diagnostics(source_path) if source_path else None
    dub_diag = audio_diagnostics(dub_path) if dub_path else None
    result = {"sample_id": row.get("sample_id", "")}
    if source_diag:
        result.update({f"source_{k}": v for k, v in source_diag.items() if k != "path"})
    if dub_diag:
        result.update({f"dub_{k}": v for k, v in dub_diag.items() if k != "path"})
    if source_diag and dub_diag and "duration_seconds" in source_diag and "duration_seconds" in dub_diag:
        drift = dub_diag["duration_seconds"] - source_diag["duration_seconds"]
        result["duration_drift_seconds"] = drift
        result["duration_drift_percent"] = 100 * drift / source_diag["duration_seconds"] if source_diag["duration_seconds"] else np.nan
    audio_rows.append(result)

audio_report = pd.DataFrame(audio_rows)
display(audio_report.head(20))
if not audio_report.empty:
    audio_report.to_csv(EVAL_DIR / "audio_diagnostics.csv", index=False)
    print("Saved:", EVAL_DIR / "audio_diagnostics.csv")


## 8. Timeline collision check for generated segments

If Vocult saves one generated file per segment and has start/end timestamps, create `evaluation/segments_timeline.csv` with columns:
`sample_id, speaker_id, start_seconds, end_seconds, generated_audio_path`.

This checks temporal collisions between generated segments assigned to different speakers. Some overlap may be intentional in the source; compare flagged collisions against source diarization before treating them as bugs.

In [ ]:
TIMELINE_PATH = EVAL_DIR / "segments_timeline.csv"

if TIMELINE_PATH.exists():
    timeline = pd.read_csv(TIMELINE_PATH)
    needed = {"sample_id", "speaker_id", "start_seconds", "end_seconds"}
    missing = needed - set(timeline.columns)
    if missing:
        print("Timeline CSV is missing columns:", sorted(missing))
    else:
        timeline = timeline.dropna(subset=["start_seconds", "end_seconds"]).copy()
        timeline["start_seconds"] = pd.to_numeric(timeline["start_seconds"], errors="coerce")
        timeline["end_seconds"] = pd.to_numeric(timeline["end_seconds"], errors="coerce")
        timeline = timeline.dropna(subset=["start_seconds", "end_seconds"])
        timeline = timeline[timeline["end_seconds"] > timeline["start_seconds"]]
        collisions = []
        for sample_id, group in timeline.groupby("sample_id"):
            records = group.sort_values("start_seconds").to_dict("records")
            for i, a in enumerate(records):
                for b in records[i + 1:]:
                    if b["start_seconds"] >= a["end_seconds"]:
                        break
                    if str(a["speaker_id"]) != str(b["speaker_id"]):
                        overlap = min(a["end_seconds"], b["end_seconds"]) - max(a["start_seconds"], b["start_seconds"])
                        if overlap > 0:
                            collisions.append({
                                "sample_id": sample_id,
                                "speaker_a": a["speaker_id"],
                                "speaker_b": b["speaker_id"],
                                "overlap_seconds": overlap,
                                "a_start": a["start_seconds"], "a_end": a["end_seconds"],
                                "b_start": b["start_seconds"], "b_end": b["end_seconds"],
                            })
        collision_df = pd.DataFrame(collisions)
        display(collision_df)
        collision_df.to_csv(EVAL_DIR / "timeline_collisions.csv", index=False)
        print(f"Flagged cross-speaker collisions: {len(collision_df)}")
        print("Saved:", EVAL_DIR / "timeline_collisions.csv")
else:
    print("No timeline CSV found. Create it from your generated segment metadata to check cross-speaker overlaps.")


## 9. Optional: speaker embedding similarity

If you have source reference clips and generated dub clips per speaker, use a speaker-verification model (e.g. SpeechBrain ECAPA-TDNN) to calculate cosine similarity between embeddings. This can help assess voice identity consistency, but it is not equivalent to speaker recognition accuracy, and cross-language / emotional changes can affect scores. Install and configure the model separately; do not treat a threshold as universal.

In [ ]:
# Optional template — requires SpeechBrain and a source/dub clip pair for each speaker.
# %pip install speechbrain torchaudio
#
# from speechbrain.inference.speaker import EncoderClassifier
# import torch, torchaudio
#
# encoder = EncoderClassifier.from_hparams(
#     source="speechbrain/spkrec-ecapa-voxceleb",
#     savedir="pretrained_models/spkrec-ecapa-voxceleb"
# )
#
# def speaker_embedding(path):
#     signal, sr = torchaudio.load(str(path))
#     if signal.shape[0] > 1:
#         signal = signal.mean(dim=0, keepdim=True)
#     if sr != 16000:
#         signal = torchaudio.functional.resample(signal, sr, 16000)
#     return encoder.encode_batch(signal).squeeze().detach().cpu().numpy()
#
# def cosine_similarity(a, b):
#     return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))
#
# # Compare source speaker-reference clip against the generated clip for that same speaker.
# # Treat the score as a relative measure across your test set, not a universal pass/fail threshold.


## 10. Export a combined metric summary

This summary records only metrics that can be computed from available annotations. Empty or unavailable metrics should remain unavailable rather than being filled with zero.

In [ ]:
summary = {
    "evaluation_rows": int(len(df)),
    "asr_segments_available": int(len(asr_pairs)),
    "translation_segments_available": int(len(translation_pairs)),
    "speaker_segments_available": int(len(speaker_pairs)),
    "emotion_segments_available": int(len(emotion_pairs)),
}

try:
    if len(asr_pairs):
        from jiwer import wer, cer
        summary["wer"] = float(wer(asr_pairs["reference_transcript"].astype(str).tolist(),
                                   asr_pairs["predicted_transcript"].astype(str).tolist()))
        summary["cer"] = float(cer(asr_pairs["reference_transcript"].astype(str).tolist(),
                                   asr_pairs["predicted_transcript"].astype(str).tolist()))
except ImportError:
    pass

try:
    if len(translation_pairs):
        import sacrebleu
        refs = translation_pairs["reference_translation"].astype(str).tolist()
        preds = translation_pairs["predicted_translation"].astype(str).tolist()
        summary["bleu"] = float(sacrebleu.corpus_bleu(preds, [refs]).score)
        summary["chrf"] = float(sacrebleu.corpus_chrf(preds, [refs]).score)
except ImportError:
    pass

try:
    from sklearn.metrics import accuracy_score, precision_recall_fscore_support
    if len(speaker_pairs):
        yt, yp = speaker_pairs["reference_speaker"].astype(str), speaker_pairs["predicted_speaker"].astype(str)
        summary["speaker_label_accuracy_after_id_mapping"] = float(accuracy_score(yt, yp))
    if len(emotion_pairs):
        yt = emotion_pairs["reference_emotion"].astype(str).str.lower().str.strip()
        yp = emotion_pairs["predicted_emotion"].astype(str).str.lower().str.strip()
        summary["emotion_accuracy"] = float(accuracy_score(yt, yp))
        summary["emotion_macro_f1"] = float(precision_recall_fscore_support(yt, yp, average="macro", zero_division=0)[2])
except ImportError:
    pass

summary_path = EVAL_DIR / "metrics_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print("Saved:", summary_path)


## 11. Recommended evaluation protocol

1. Select a representative set of clips with different speakers, accents, speaking rates, and emotions.
2. Manually verify source transcripts, target translations, speaker turns, and emotion labels.
3. Run the exact Vocult pipeline version being evaluated and record its outputs in the manifest.
4. Report each metric with its sample count and test-set description.
5. Listen to the output and conduct a small human rating study for **naturalness, emotion preservation, intelligibility, voice identity, and synchronization**. Automatic metrics alone cannot fully measure these qualities.
6. For the current overlap bug, prioritize the timeline collision report and listening checks. For emotion expression, evaluate human-rated source/dub emotion similarity and prosody, not just emotion-classifier accuracy.

### Interpreting results
- **WER / CER:** lower is better.
- **BLEU / chrF:** higher is generally better, but compare only across the same dataset and tokenization/evaluation setup.
- **Accuracy / precision / recall / F1:** higher is better, but class imbalance can make accuracy misleading; inspect macro-F1 and the confusion matrix.
- **DER:** lower is better; calculate with a proper diarization evaluation tool and RTTM annotations.
- **Duration drift / clipping / silence:** interpret against your expected delivery and source media. There is no universal threshold that guarantees quality.
